# 08. Capstone Project: End-to-End PySpark Data Pipeline

This notebook brings together the key ideas from the series into a realistic data engineering project.

## Project objective
Build an end-to-end ETL pipeline for retail or sales data using PySpark.

## Project scope
- read raw CSV input
- clean invalid data
- transform to analytics-ready schema
- aggregate KPIs
- save output in Parquet
- generate summary dataset


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('CapstonePipeline').master('local[*]').getOrCreate()

raw = spark.createDataFrame([
    ('2024-01-01', 'A', 1200),
    ('2024-01-02', 'A', 1500),
    ('2024-01-01', 'B', 900),
    ('2024-01-03', 'B', 1700),
    ('2024-01-02', 'C', 800),
], ['date', 'region', 'sales'])

cleaned = raw.withColumn('sales', F.col('sales').cast('double'))
cleaned = cleaned.filter(F.col('sales').isNotNull())

summary = cleaned.groupBy('region').agg(
    F.sum('sales').alias('total_sales'),
    F.avg('sales').alias('avg_sales'),
    F.count('sales').alias('sales_records')
)

summary.show()
cleaned.write.mode('overwrite').parquet('data/output/capstone_sales_cleaned.parquet')
summary.write.mode('overwrite').parquet('data/output/capstone_sales_summary.parquet')

spark.stop()


## Reflection questions

- What parts of the pipeline are transformations?
- Where would you use `cache()`?
- How would you handle larger files or skewed partitions?
- How would you move this logic to BigQuery or GCS in production?

## Suggested next steps

1. Add a `date` parsing step
2. Add customer or product dimension tables
3. Build a report with monthly KPIs
4. Use Spark SQL and compare with DataFrame API
5. Put the job into a command-line script or Airflow DAG
